In [1]:
import pandas as pd

In [2]:
path = 'data/model_annotations_aligned.paired.jsonl'

In [7]:
df = pd.read_json(path, lines=True)

print('Length of data: ', len(df))

print(df.head(2))

Length of data:  1600
                                                 id  \
0  dm-test-8764fb95bfad8ee849274873a92fb8d6b400eee2   
1  dm-test-8764fb95bfad8ee849274873a92fb8d6b400eee2   

                                             decoded  \
0  paul merson was brought on with only seven min...   
1  paul merson has restarted his row with andros ...   

                                  expert_annotations  \
0  [{'coherence': 2, 'consistency': 1, 'fluency':...   
1  [{'coherence': 3, 'consistency': 5, 'fluency':...   

                                  turker_annotations  \
0  [{'coherence': 3, 'consistency': 3, 'fluency':...   
1  [{'coherence': 2, 'consistency': 3, 'fluency':...   

                                          references model_id  \
0  [Andros Townsend an 83rd minute sub in Tottenh...      M11   
1  [Andros Townsend an 83rd minute sub in Tottenh...      M13   

                                            filepath  \
0  cnndm/dailymail/stories/8764fb95bfad8ee8492748... 

In [8]:
# Create the first DataFrame: `data`
data = pd.DataFrame({
    'input': df['text'],
    'actual_output': df['decoded']
})

data.head(2)

,input,actual_output
0,Paul Merson has restarted his row with Andros ...,paul merson was brought on with only seven min...
1,Paul Merson has restarted his row with Andros ...,paul merson has restarted his row with andros ...


In [10]:
annotations = pd.DataFrame({
    'expert_annotations': df['expert_annotations'],
    'turker_annotations': df['turker_annotations']
})

annotations.head(2)

,expert_annotations,turker_annotations
0,"[{'coherence': 2, 'consistency': 1, 'fluency':...","[{'coherence': 3, 'consistency': 3, 'fluency':..."
1,"[{'coherence': 3, 'consistency': 5, 'fluency':...","[{'coherence': 2, 'consistency': 3, 'fluency':..."


In [11]:
# Save both DataFrames to CSV
data.to_csv('data/data.csv', index=False)
annotations.to_csv('data/annotations.csv', index=False)

In [7]:
import pandas as pd
import ast

file_order = ['outputs_old/de_summarization_scores_1-500.csv', 
              'outputs_old/de_summarization_scores_500-1000.csv', 
              'outputs_old/de_summarization_scores_1000-1600.csv']

# List to store filtered DataFrames
dfs = []

# Offset to shift index values
offset = 0

for file in file_order:
    # Read the CSV file
    df = pd.read_csv(file)

    # Filter rows where success is True
    df = df[df['success'] == True]

    # Adjust index column to be continuous
    df['index'] = df['index'] + offset

    # Parse 'score_bd' to extract alignment and coverage
    df['alignment'] = df['score_bd'].apply(lambda x: ast.literal_eval(x)['Alignment'])
    df['coverage'] = df['score_bd'].apply(lambda x: ast.literal_eval(x)['Coverage'])

    # Keep only required columns
    df = df[['index', 'score', 'alignment', 'coverage']]

    # Append to list
    dfs.append(df)

    # Update offset for next file
    offset += 500

# Concatenate all data
combined_df = pd.concat(dfs, ignore_index=True)

combined_df.to_csv('data/deepeval.csv', index=False)

In [8]:
import pandas as pd

file_order = ['outputs_old/geval_summarization_scores_1-500.csv',
              'outputs_old/geval_summarization_scores_500-1000.csv', 
              'outputs_old/geval_summarization_scores_1000-1600.csv']

dfs = []

offset = 0

for file in file_order:
    # Read the CSV file
    df = pd.read_csv(file)

    # Filter only successful rows
    df = df[df['success'] == True]

    # Adjust index
    df['index'] = df['index'] + offset

    # Keep only required columns
    df = df[['index', 'coherence_score', 'consistency_score', 'fluency_score', 'relevance_score', 'average_score']]

    # Append to list
    dfs.append(df)

    # Update offset
    offset += 500

combined_df = pd.concat(dfs, ignore_index=True)
combined_df.to_csv('data/geval.csv', index=False)

In [1]:
import pandas as pd
import ast

# Read CSV
df = pd.read_csv("data/annotations.csv")  # Replace with your file name

rows = []
for i, row in df.iterrows():
    # Safely evaluate the stringified lists of dicts
    expert = ast.literal_eval(row['expert_annotations'])
    turker = ast.literal_eval(row['turker_annotations'])

    # Combine expert + turker annotations
    all_anns = expert + turker

    # Compute average scores
    coherence = sum(d['coherence'] for d in all_anns) / len(all_anns)
    consistency = sum(d['consistency'] for d in all_anns) / len(all_anns)
    fluency = sum(d['fluency'] for d in all_anns) / len(all_anns)
    relevance = sum(d['relevance'] for d in all_anns) / len(all_anns)

    rows.append([i, coherence, consistency, fluency, relevance])

# Create new DataFrame
result_df = pd.DataFrame(rows, columns=['index', 'coherence', 'consistency', 'fluency', 'relevance'])

# Save to CSV
result_df.to_csv("data/ann.csv", index=False)